# Which UBC docks change fastest, and does the school calendar improve the warning?

A rider can care about three different things: how much stock moves at a dock, how many trips people actually take, and how often the last e-bike disappears during a walk. We measure all three. We rank the 17 stations on an earlier common winter, then test five- and ten-minute warnings at every station across eight later months. Brock Commons is one comparison dock.

Use September 1 through April 30 in Vancouver as the school-year approximation. Finals, holidays and reading breaks are included. The precision experiment compares this calendar feature with the same model and inputs otherwise. These archive replays are exploratory and their walks are simulated.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/prepare_data.py").exists())
sys.path.insert(0, str(root / "notebooks"))
from availability_tools import covered_times, station_decisions, forward_split, predict, warning_weights

TZ = "America/Vancouver"
RANK_FROM = pd.Timestamp("2025-12-01", tz=TZ)
MONTHS = pd.date_range("2026-03-01", "2026-10-01", freq="MS", tz=TZ)
WALKS = (5, 10)
RATE = 0.20
def in_school(month):
    return (month >= 9) | (month <= 4)

## 1. Choose docks using inventory movement before the replay

Measure the sum of absolute changes in consecutive published e-bike counts per 24 hours of covered archive time. A reported change from 2 to 5 contributes three. This captures both positive and negative movement, including possible rebalancing. Two opposite movements between reports can cancel. Separate changes across citywide archive outages and discard impossible counts. All 17 stations had opened before the ranking window.

In [2]:
data = root / "data/processed"
inventory = pd.read_parquet(data / "ubc_inventory.parquet").sort_values(["station_id", "observed_at"])
network = pd.DatetimeIndex(pd.read_parquet(data / "network_observations.parquet").observed_at).sort_values().unique()
end = MONTHS[-1] + pd.offsets.MonthBegin(1)
inventory = inventory.loc[inventory.observed_at < end].copy()
network = network[network < end]
assert not inventory.duplicated(["station_id", "observed_at"]).any()
assert inventory.observed_at.isin(network).all()
bad = inventory.ebikes.lt(0) | inventory.ebikes.gt(inventory.total_bikes)
inventory.loc[bad.fillna(False), "ebikes"] = pd.NA
names = inventory.groupby("station_id").station_name.last()
first_seen = inventory.groupby("station_id").observed_at.min()
assert len(names) == 17 and first_seen.max() < RANK_FROM

In [3]:
cycle = network.to_series().diff().gt(pd.Timedelta(minutes=10)).cumsum()
inv = inventory.copy()
inv["segment"] = inv.observed_at.map(cycle)
inv["open_count"] = inv.ebikes.where(inv.is_renting.ne(False).fillna(True))
inv["step"] = inv.groupby(["station_id", "segment"]).open_count.diff().abs()
inv["bin"] = inv.observed_at.dt.floor("10min")
inv["day"] = inv.observed_at.dt.tz_convert(TZ).dt.normalize()

# Only count activity in ten-minute windows that the citywide archive spans.
bins = covered_times(network, interval_min=10, history_min=0, arrival_min=10, max_gap_min=10).time
hours = bins.groupby(bins.dt.tz_convert(TZ).dt.normalize()).size().div(6).rename("hours")
seen = inv.loc[inv.bin.isin(bins) & inv.open_count.notna()]
daily = seen.groupby(["station_id", "day"]).step.agg(movement="sum", known_steps="count").join(hours, on="day")
daily = daily.loc[(daily.hours >= 20) & (daily.known_steps > 0)].reset_index()
print("Shared ranking window:", RANK_FROM.date(), "to", (MONTHS[0] - pd.Timedelta(days=1)).date())

Shared ranking window: 2025-12-01 to 2026-02-28


In [4]:
def movement_rate(days):
    totals = days.groupby("station_id").agg(days=("day", "size"), movement=("movement", "sum"), hours=("hours", "sum"))
    return totals.assign(movement_per_day=24 * totals.movement / totals.hours)

winter = daily.loc[daily.day.between(RANK_FROM, MONTHS[0], inclusive="left")]
# Give all stations the same dates, dropping a day for all if one station had no comparable observations.
complete = winter.groupby("day").station_id.nunique().loc[lambda n: n == len(names)].index
ranks = movement_rate(winter.loc[winter.day.isin(complete)]).sort_values("movement_per_day", ascending=False)
assert len(ranks) == 17 and len(complete) >= 30
ranks["rank"] = np.arange(1, len(ranks) + 1)
ranks["group"] = pd.cut(ranks["rank"], bins=[0, 6, 12, 17], labels=["High", "Middle", "Low"])
ranks["station"] = ranks.index.map(names)
display(ranks[["rank", "station", "group", "days", "movement_per_day"]].round(2))

,rank,station,group,days,movement_per_day
station_id,,,,,
2117,1,Totem Park,High,47,28.85
2116,2,Orchard Commons,High,47,26.29
2134,3,UBC Bookstore,High,47,23.22
2132,4,IK Barber Learning Centre,High,47,21.94
2252,5,Wesbrook Village - Berton & Shrum,High,47,21.36
2143,6,War Memorial Gym,High,47,19.48
2136,7,Agronomy & East Mall,Middle,47,17.01
2114,8,West Mall Swing Space Building,Middle,47,16.87
2131,9,Brock Commons,Middle,47,14.86


Keep those groups fixed for the remainder. Check whether the ranking survives the change of season. Apply the same coverage rule and compare all stations on the same dates within each period. A station can change groups in practice even though its pilot group is frozen.

The common window yielded 47 comparable dates. Totem Park (28.85 observed e-bikes of movement per covered day) and Orchard Commons (26.29) led. Brock Commons was ninth at 14.86. Check how much requiring every dock to report on the same dates affects this order.

In [5]:
available = movement_rate(winter)
sensitivity = ranks[["station", "rank", "days"]].rename(columns={"days": "shared days"})
sensitivity["individual days"] = available.days
sensitivity["rank using individual days"] = available.movement_per_day.rank(ascending=False, method="min").astype(int)
display(sensitivity)

,station,rank,shared days,individual days,rank using individual days
station_id,,,,,
2117,Totem Park,1,47,57,1
2116,Orchard Commons,2,47,63,2
2134,UBC Bookstore,3,47,61,5
2132,IK Barber Learning Centre,4,47,59,4
2252,Wesbrook Village - Berton & Shrum,5,47,63,3
2143,War Memorial Gym,6,47,62,6
2136,Agronomy & East Mall,7,47,57,8
2114,West Mall Swing Space Building,8,47,59,7
2131,Brock Commons,9,47,62,9


In [6]:
later = daily.loc[daily.day.between(MONTHS[0], end, inclusive="left")].copy()
later = later.loc[later.day.isin(later.groupby("day").station_id.nunique().loc[lambda n: n == 17].index)]
later["period"] = np.where(in_school(later.day.dt.month), "school", "summer")
stability = ranks[["rank", "station", "group", "movement_per_day"]].copy()
for period, days in later.groupby("period"):
    rates = movement_rate(days).movement_per_day
    stability[period] = rates
    stability[period + "_rank"] = rates.rank(ascending=False, method="min").astype(int)
display(stability.round(1))
print("Common later dates:", later.groupby("period").day.nunique().to_dict())

,rank,station,group,movement_per_day,school,school_rank,summer,summer_rank
station_id,,,,,,,,
2117,1,Totem Park,High,28.9,28.7,5,4.9,16
2116,2,Orchard Commons,High,26.3,30.4,2,8.4,14
2134,3,UBC Bookstore,High,23.2,29.8,3,15.4,6
2132,4,IK Barber Learning Centre,High,21.9,29.2,4,10.3,10
2252,5,Wesbrook Village - Berton & Shrum,High,21.4,31.4,1,28.2,1
2143,6,War Memorial Gym,High,19.5,23.4,8,20.5,5
2136,7,Agronomy & East Mall,Middle,17.0,19.8,10,20.7,4
2114,8,West Mall Swing Space Building,Middle,16.9,24.2,6,23.5,2
2131,9,Brock Commons,Middle,14.9,24.1,7,22.3,3


Common later dates: {'school': 109, 'summer': 34}


## 2. Count rider departures across an entire school year and summer

Inventory changes also include staff movements. Use the operator's monthly trip files for the use question. Count departures at campus docks once per trip, separately for all bikes and e-bikes. Use the 16 docks already observed on September 1, 2025, and keep that panel fixed until August 31, 2026. Normalize by calendar days to compare eight school months with four summer months. The files report trips by publishing month. The operator removes staff rebalancing trips and rounds times to the hour. [System data and data notes](https://www.mobibikes.ca/en/system-data)

In [7]:
trip_dir = root / "data/trip_counts"
trip_months = pd.period_range("2025-09", "2026-08", freq="M")
if any(not (trip_dir / f"{month}.csv").exists() for month in trip_months):
    raise FileNotFoundError("Run python scripts/prepare_trip_counts.py before opening this notebook.")
parts = [pd.read_csv(trip_dir / f"{month}.csv", dtype={"station_id": str}).assign(month=month)
         for month in trip_months]
trips = pd.concat(parts, ignore_index=True)
panel = first_seen.index[first_seen < pd.Timestamp("2025-09-01", tz=TZ)]
assert len(panel) == 16 and len(trips) == 12 * len(names)
monthly = trips.loc[trips.station_id.isin(panel)].groupby("month")[["departures", "ebike_departures"]].sum()
monthly["days"] = monthly.index.days_in_month
monthly["period"] = np.where(in_school(monthly.index.month), "School", "Summer")
display(monthly.assign(ebike_starts_per_day=monthly.ebike_departures / monthly.days).round(1))

,departures,ebike_departures,days,period,ebike_starts_per_day
month,,,,,
2025-09,7002,3945,30,School,131.5
2025-10,7307,5666,31,School,182.8
2025-11,5809,4571,30,School,152.4
2025-12,3345,2729,31,School,88.0
2026-01,5066,4163,31,School,134.3
2026-02,4290,3393,28,School,121.2
2026-03,5601,4250,31,School,137.1
2026-04,5521,4070,30,School,135.7
2026-05,3998,2527,31,Summer,81.5


In [8]:
season = monthly.groupby("period")[["departures", "ebike_departures", "days"]].sum()
season["trips_per_day"] = season.departures / season.days
season["ebike_starts_per_day"] = season.ebike_departures / season.days
display(season.round(1))
print("School-to-summer e-bike departure ratio:", round(season.loc["School","ebike_starts_per_day"] /
                                                        season.loc["Summer","ebike_starts_per_day"], 2))

,departures,ebike_departures,days,trips_per_day,ebike_starts_per_day
period,,,,,
School,43941,32787,242,181.6,135.5
Summer,20437,12675,123,166.2,103.0


School-to-summer e-bike departure ratio: 1.31


The same campus total can conceal a strong summer drop at one dock and an increase at another. Compare actual e-bike starts per calendar day for each dock in the fixed panel. These trip counts also give us a cross-check on the seasonal inventory ranking, which used fewer common summer dates.

In [9]:
local_trips = trips.loc[trips.station_id.isin(panel)].copy()
local_trips["period"] = np.where(in_school(local_trips.month.dt.month), "School", "Summer")
starts = local_trips.groupby(["station_id", "period"]).ebike_departures.sum().unstack()
for period in ("School", "Summer"):
    starts[period] = starts[period] / season.loc[period, "days"]
starts["school/summer"] = starts.School / starts.Summer
display(ranks[["rank", "station"]].join(starts, how="inner").round(2))

,rank,station,School,Summer,school/summer
station_id,,,,,
2117,1,Totem Park,15.48,2.55,6.06
2116,2,Orchard Commons,12.43,2.86,4.34
2134,3,UBC Bookstore,11.05,6.81,1.62
2132,4,IK Barber Learning Centre,10.25,4.82,2.13
2252,5,Wesbrook Village - Berton & Shrum,11.54,13.39,0.86
2143,6,War Memorial Gym,8.70,9.51,0.91
2136,7,Agronomy & East Mall,6.71,7.13,0.94
2114,8,West Mall Swing Space Building,9.33,9.76,0.96
2131,9,Brock Commons,8.12,8.49,0.96


Actual e-bike departures at Totem Park averaged 15.48 per school-year day and 2.55 per summer day. Wesbrook Village moved in the other direction, 11.54 versus 13.39. Orchard Commons had zero departures in the operator’s June 2026 file. The total describes realized use; service changes and seasonal demand can both affect it. Repeat the campus comparison without Orchard to see how much it matters.

In [10]:
without_orchard = local_trips.loc[local_trips.station_id.ne("2116")].groupby("period").ebike_departures.sum()
sensitivity_daily = without_orchard / season.days
display(sensitivity_daily.round(2).to_frame("daily e-bike departures, excluding Orchard Commons"))
print("School/summer ratio without Orchard:", round(sensitivity_daily.School / sensitivity_daily.Summer, 2))

,"daily e-bike departures, excluding Orchard Commons"
period,
School,123.06
Summer,100.19


School/summer ratio without Orchard: 1.23


## 3. Does a busier dock also lose its last bike more often?

Reuse the original five-minute wall-clock replay, its ten-minute station-report freshness limit and its outage filters. A walk starts with positive inventory and no reported rental closure. At the deadline, either a reported zero or an explicit closure counts as an empty arrival. When comparing warnings at each dock, restrict to the cases in which exactly one e-bike was available at departure.

In [11]:
grid = covered_times(network, interval_min=5, history_min=15, arrival_min=10, max_gap_min=10)
walks = pd.concat([station_decisions(group, grid, WALKS, freshness_min=10)
                   for _, group in inventory.groupby("station_id")], ignore_index=True)
for h in WALKS:
    count, closed = walks[f"arrival_ebikes_{h}"], walks[f"arrival_closed_{h}"]
    walks[f"empty_{h}"] = (closed | count.eq(0)).where(closed | count.notna())
walks = walks.dropna(subset=[f"empty_{h}" for h in WALKS]).sort_values("time")
local = walks.time.dt.tz_convert(TZ)
walks["school"] = in_school(local.dt.month).astype(int)
walks["weekend"] = local.dt.dayofweek.ge(5).astype(int)
angle = 2 * np.pi * (local.dt.hour + local.dt.minute / 60) / 24
walks["hour_sin"], walks["hour_cos"] = np.sin(angle), np.cos(angle)
assert (walks.observed_at <= walks.time).all()

In [12]:
splits = {m: forward_split(walks, m, max(WALKS)) for m in MONTHS}
one = pd.concat([test.loc[test.ebikes.eq(1)] for _, test in splits.values()])
risk = one.groupby("station_id").agg(one_bike_walks=("empty_10", "size"), empty=("empty_10", "sum"))
risk["empty_in_10_pct"] = 100 * risk["empty"].astype(int) / risk.one_bike_walks
display(ranks[["rank", "station", "group", "movement_per_day"]].join(risk).round(2))

,rank,station,group,movement_per_day,one_bike_walks,empty,empty_in_10_pct
station_id,,,,,,,
2117,1,Totem Park,High,28.85,2589,363,14.02
2116,2,Orchard Commons,High,26.29,2690,558,20.74
2134,3,UBC Bookstore,High,23.22,2826,620,21.94
2132,4,IK Barber Learning Centre,High,21.94,2385,392,16.44
2252,5,Wesbrook Village - Berton & Shrum,High,21.36,3473,562,16.18
2143,6,War Memorial Gym,High,19.48,3364,506,15.04
2136,7,Agronomy & East Mall,Middle,17.01,1607,164,10.21
2114,8,West Mall Swing Space Building,Middle,16.87,2849,364,12.78
2131,9,Brock Commons,Middle,14.86,2735,411,15.03


A station can have zero eligible one-bike walks in a month. Show these coverage gaps explicitly and omit such station-months from scoring. Every station must still appear somewhere in the eight-month replay.

In [13]:
availability = pd.DataFrame({m.strftime("%Y-%m"): v.loc[v.ebikes.eq(1)].groupby("station_id").size()
                             for m, (_, v) in splits.items()}).reindex(names.index).fillna(0).astype(int)
assert availability.sum(axis=1).gt(0).all()
missing = availability.eq(0).stack()
print("Observed station-months:", int((~missing).sum()), "of", len(missing))
print("No eligible one-bike walks:", [(names.loc[station], month) for station, month in missing.index[missing]])

Observed station-months: 135 of 136
No eligible one-bike walks: [('Orchard Commons', '2026-06')]


## 4. Does knowing the school year improve a warning at the same dock?

Use the shallow tree from the earlier ten-minute experiment as the common reference at both horizons. Fit from scratch at each Vancouver month boundary, using only decisions whose longer walk has finished. Evaluate March–October 2026, four school months and four summer months. The forecast inputs include station, e-bike and total-bike counts, clock and weekend. Compare the same tree with a school flag, then with explicit school-by-station and school-by-clock interactions. All three versions receive identical training examples and warning budgets.

One of 136 station-months has no eligible walks (Orchard Commons in June). October's last local day is partial because the inventory archive ends at midnight UTC.

In [14]:
walks["school_station"] = walks.station_id + np.where(walks.school.eq(1), "_school", "_summer")
walks["school_sin"] = walks.school * walks.hour_sin
walks["school_cos"] = walks.school * walks.hour_cos
walks["school_weekend"] = walks.school * walks.weekend
base = ["station_id", "ebikes", "total_bikes", "hour_sin", "hour_cos", "weekend"]
feature_sets = {"Clock": base, "School flag": base + ["school"],
                "School interactions": base + ["school", "school_station", "school_sin", "school_cos", "school_weekend"]}

def estimator(fields):
    categories = [f for f in fields if f in ("station_id", "school_station")]
    prep = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categories),
                              ("numbers", "passthrough", [f for f in fields if f not in categories])])
    return make_pipeline(prep, HistGradientBoostingClassifier(max_depth=3, max_iter=80, learning_rate=.07,
                         min_samples_leaf=40, l2_regularization=1, early_stopping=False, random_state=42))

In [15]:
def replay(fields):
    parts = []
    for month in MONTHS:
        train, test = forward_split(walks, month, max(WALKS))
        test = test.loc[test.ebikes.eq(1)]
        assert train.time.max() + pd.Timedelta(minutes=max(WALKS)) < month
        assert (test.time >= month).all() and not test.empty
        assert (test.time + pd.Timedelta(minutes=max(WALKS)) < month + pd.offsets.MonthBegin(1)).all()
        for h in WALKS:
            scores = predict(estimator(fields), fields, train, test, h)
            parts.append(pd.DataFrame({"time": test.time, "station_id": test.station_id, "month": month,
                         "h": h, "y": test[f"empty_{h}"].astype(int), "score": scores}))
    return pd.concat(parts, ignore_index=True)

def warn(frame):
    frame = frame.copy()
    # Each dock gets its own 20% budget each month and at each walking time.
    frame["w"] = frame.groupby(["station_id", "month", "h"]).score.transform(
        lambda x: warning_weights(x.to_numpy(), RATE))
    frame["hits"] = frame.w * frame.y
    frame["squared_error"] = (frame.score - frame.y) ** 2
    return frame

def summary(frame, by):
    totals = frame.groupby(by, observed=True).agg(walks=("y","size"), empty=("y","sum"),
                warnings=("w","sum"), hits=("hits","sum"), brier=("squared_error","mean"))
    return totals.assign(precision=100 * totals.hits / totals.warnings,
                         empty_rate=100 * totals["empty"] / totals.walks, recall=100 * totals.hits / totals["empty"])

Start with the clock reference. The prevalence column also gives the precision of randomly breaking ties among one-bike walks at each dock. Brier score measures the full probability forecast, which can change even if a model ranks walks in the same order.

In [16]:
forecasts = {}
def run_variant(name):
    forecasts[name] = warn(replay(feature_sets[name]))
    return summary(forecasts[name], "h")[["walks", "empty_rate", "precision", "recall", "brier"]].round(4)

run_variant("Clock")

,walks,empty_rate,precision,recall,brier
h,,,,,
5,41045,8.0521,10.9777,27.2666,0.0741
10,41045,13.5851,19.0070,27.9822,0.1177


Add one binary feature. Every walk in a given month has the same flag. A tree can still learn different station or hour effects across seasons, but a uniform probability shift alone cannot improve the station's within-month ranking.

In [17]:
run_variant("School flag")

,walks,empty_rate,precision,recall,brier
h,,,,,
5,41045,8.0521,10.8995,27.0722,0.0742
10,41045,13.5851,18.9528,27.9024,0.1178


Finally give the tree a dock-and-season category and local clock values interacted with the school term. This allows a residence hub and a teaching-building hub to have different school-year changes. Keep every split and model hyperparameter fixed.

The simple flag moved ten-minute warning precision from 19.01% to 18.95%. Demand can vary sharply by season without a binary flag helping rank the risky walks within each dock and month.

In [18]:
run_variant("School interactions")

,walks,empty_rate,precision,recall,brier
h,,,,,
5,41045,8.0521,10.8924,27.0546,0.0742
10,41045,13.5851,19.1257,28.1569,0.1178


## 5. Compare the same walks by dock, season and horizon

Pool correct warnings and warning weight across station-months. A busy station therefore contributes more eligible walks. The station table keeps each individual dock visible. The monthly and group tables expose effects that a campus average can hide.

In [19]:
results = pd.concat(forecasts, names=["variant"]).reset_index(level=0)
results["period"] = np.where(in_school(results.month.dt.month), "School", "Summer")
results["group"] = results.station_id.map(ranks.group)
for name, candidate in forecasts.items():
    assert candidate[["time", "station_id", "h", "y"]].equals(forecasts["Clock"][["time", "station_id", "h", "y"]])
ten = results.loc[results.h.eq(10)]
by_group = summary(ten, ["group", "period", "variant"]).reset_index()
tab = by_group.pivot(index=["group", "period"], columns="variant", values="precision")
support = by_group.loc[by_group.variant.eq("Clock")].set_index(["group", "period"])
display(support[["walks","empty","empty_rate"]].join(tab).round(2))
monthly_scores = summary(ten, ["month","variant"]).precision.unstack()
monthly_scores.index = monthly_scores.index.strftime("%b")
display(monthly_scores.round(2))

walks  empty  empty_rate  Clock  School flag  \
group  period                                                 
High   School  10790   1963       18.19  23.73        23.86   
       Summer   6537   1038       15.88  22.51        22.29   
Middle School   7492    957       12.77  18.73        18.44   
       Summer   6994    863       12.34  18.33        18.21   
Low    School   5353    509        9.51  12.38        12.40   
       Summer   3879    246        6.34  10.88        11.05   

               School interactions  
group  period                       
High   School                24.19  
       Summer                22.79  
Middle School                19.08  
       Summer                17.73  
Low    School                12.44  
       Summer                10.70

variant,Clock,School flag,School interactions
month,,,
Mar,12.62,12.56,12.93
Apr,14.87,14.21,14.05
May,18.02,17.88,17.92
Jun,15.62,15.72,16.18
Jul,19.40,19.12,18.18
Aug,19.34,19.31,19.48
Sep,24.46,24.58,25.28
Oct,21.94,22.15,22.51


In [20]:
by_station = summary(ten, ["station_id", "variant"]).reset_index()
dock_table = by_station.pivot(index="station_id", columns="variant", values="precision")
display(ranks[["rank", "station", "group"]].join(risk[["one_bike_walks", "empty_in_10_pct"]])
        .join(dock_table).round(2))

,rank,station,group,one_bike_walks,empty_in_10_pct,Clock,School flag,School interactions
station_id,,,,,,,,
2117,1,Totem Park,High,2589,14.02,14.57,16.33,15.58
2116,2,Orchard Commons,High,2690,20.74,27.40,26.07,26.46
2134,3,UBC Bookstore,High,2826,21.94,39.12,38.81,37.43
2132,4,IK Barber Learning Centre,High,2385,16.44,23.83,23.36,25.02
2252,5,Wesbrook Village - Berton & Shrum,High,3473,16.18,16.10,15.82,15.62
2143,6,War Memorial Gym,High,3364,15.04,20.34,20.92,23.40
2136,7,Agronomy & East Mall,Middle,1607,10.21,20.86,21.55,21.19
2114,8,West Mall Swing Space Building,Middle,2849,12.78,18.58,17.40,18.13
2131,9,Brock Commons,Middle,2735,15.03,23.14,23.97,24.12


How large is the difference across the eight months? Double-counting adjacent walk starts as independent riders would make the result look too certain. Resample whole Vancouver calendar weeks with all stations and models kept together, retaining their original warning assignments. The intervals describe this replay and do not capture variation across different academic years.

In [21]:
def paired_weeks(reference, candidate, horizon):
    a = forecasts[reference].loc[lambda x: x.h == horizon]
    b = forecasts[candidate].loc[lambda x: x.h == horizon]
    weeks = a.time.dt.tz_convert(TZ).dt.tz_localize(None).dt.to_period("W-SUN")
    paired = pd.DataFrame({"wh": a.w, "hit": a.hits, "wh_new": b.w, "hit_new": b.hits, "week": weeks})
    totals = paired.groupby("week").sum().to_numpy()
    draws = np.random.default_rng(42).integers(len(totals), size=(2000, len(totals)))
    bootstrap = totals[draws].sum(axis=1)
    diff = lambda z: 100 * (z[...,3] / z[...,2] - z[...,1] / z[...,0])
    lo, hi = np.quantile(diff(bootstrap), [.025, .975])
    return [diff(totals.sum(axis=0)), lo, hi]

intervals = [{"variant": v, "minutes": h, **dict(zip(["gain (pp)", "low", "high"],
              paired_weeks("Clock", v, h)))} for v in ("School flag","School interactions") for h in WALKS]
display(pd.DataFrame(intervals).round(2))

,variant,minutes,gain (pp),low,high
0,School flag,5,-0.08,-0.27,0.15
1,School flag,10,-0.05,-0.33,0.20
2,School interactions,5,-0.09,-0.36,0.17
3,School interactions,10,0.12,-0.36,0.60


## Findings and follow-up

Across 41,045 one-bike walk starts, ten-minute warning precision was 19.01% with the clock, 18.95% with the school flag, and 19.13% with explicit school interactions. The interaction gain was 0.12 percentage points (paired-week 95% interval −0.36 to +0.60). There is little evidence for adding the calendar features to this warning rule. The seasonal departure totals are still useful for choosing where and when to run a pilot.

The Bookstore’s ten-minute clock warnings were correct 39.12% of the time against a 21.94% empty-arrival rate. At Totem the figures were 14.57% and 14.02%. That led to two follow-up experiments. Would suggesting a nearby backup save a wasted walk? Could timing within the hour or recent inventory changes at neighbouring docks improve the warning at Totem? Both replays are complete. [Backup walks and Totem forecasts](backup_walks.ipynb)